# Q3角度.ipynb

步態畢業專題研究筆記。此檔保留探索步驟，已清除輸出、執行資訊及 Colab 中繼資料。

包含獨立實驗與尚未統一的座標定義，不保證由上而下可執行；使用整合流程請從根目錄 README 開始。舊輸出不作為本次驗證結果。


In [ ]:
import os
import math
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
import io


In [ ]:
uploaded = files.upload()


all_data = pd.DataFrame()


In [ ]:
file_q3_angles = {}

file_avg_angles = {}


In [ ]:
def angle_change(x1, y1, x2, y2, x3, y3):
    a_x = x2 - x1
    a_y = y2 - y1
    b_x = x3 - x2
    b_y = y3 - y2
    dot_product = a_x * b_x + a_y * b_y
    magnitude_a = math.sqrt(a_x**2 + a_y**2)
    magnitude_b = math.sqrt(b_x**2 + b_y**2)
    if magnitude_a * magnitude_b == 0:
        return 0
    cos_angle = dot_product / (magnitude_a * magnitude_b)
    cos_angle = max(-1, min(1, cos_angle))
    return math.acos(cos_angle) * (180 / math.pi)


for file_name in uploaded.keys():
    print(f"Reading file: {file_name}")
    df = pd.read_csv(io.BytesIO(uploaded[file_name]))


    kpts_columns = df['kpts'].str.split(',', expand=True)


    for i in range(kpts_columns.shape[1]):
        kpts_columns.rename(columns={i: f'kpt_{i+1}'}, inplace=True)


    df = df.join(kpts_columns, rsuffix='_kpts')


    x11 = df['kpt_31'].astype(float)
    y11 = df['kpt_32'].astype(float)
    x12 = df['kpt_34'].astype(float)
    y12 = df['kpt_35'].astype(float)
    x14 = df['kpt_40'].astype(float)
    y14 = df['kpt_41'].astype(float)
    x15 = df['kpt_44'].astype(float)
    y15 = df['kpt_45'].astype(float)
    x18 = (x11 + x12) / 2
    y18 = (y11 + y12) / 2


    angle_changes = []
    for i in range(len(x18)):
        angle_change_value = angle_change(x14[i], y14[i], x18[i], y18[i], x15[i], y15[i])
        angle_changes.append(angle_change_value)


    df['Q3_angle_change'] = angle_changes


    q3_angle_change = pd.Series(angle_changes).quantile(0.75) if len(angle_changes) > 0 else 0


    file_q3_angles[file_name] = q3_angle_change


    file_avg_angles[file_name] = q3_angle_change


    all_data = pd.concat([all_data, df], ignore_index=True)


In [ ]:
q3_angle_change = all_data['Q3_angle_change'].quantile(0.75)


In [ ]:
print(f"Q3角度變化: {q3_angle_change}")


In [ ]:
output_csv_filename = 'Q3角度.csv'
all_data.to_csv(output_csv_filename, index=False)
print(f"角度變化計算完成，結果已保存到 '{output_csv_filename}'")
files.download(output_csv_filename)


In [ ]:
file_names = list(file_q3_angles.keys())
q3_angles = list(file_q3_angles.values())
avg_angles = list(file_avg_angles.values())


In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(file_names, q3_angles, color='skyblue')
plt.title('Q3 Angle Change per File')
plt.xlabel('File Name')
plt.ylabel('Q3 Angle Change')
plt.xticks(rotation=90, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
plt.boxplot([avg_angles], vert=False, patch_artist=True, boxprops=dict(facecolor='skyblue'))
plt.title('Q3 Angle Change Box Plot for Each File')
plt.xlabel('Q3 Angle Change')
plt.tight_layout()
plt.show()
